In [2]:
# %% [markdown]
# # Cathode Bending vs. proximity to cathode — reviewer comment #17 (Anne), take 2
#
# From the Part-1 tree listing you sent back, the file layout is NOT what I
# guessed the first time:
#
# - The main event tree is nested: `events/full/selected` (cycle 2 = latest),
#   plus `events/onbeam/selected`, `events/offbeam/selected`.
# - The Cathode Bending variation (var09, per dpT.toml) shows up as
#   `variations/reco_leading_muon_ke_var09_1D` and `..._2D` — these read like
#   pre-built histogram objects (TH1/TH2) keyed to one specific observable
#   (reco_leading_muon_ke), not a per-event tree.
# - There's also `detsys_results/var09` and `detsys_results/var09_splines/bin0..bin10`
#   — likely the per-truth-bin spline objects used to build the detector
#   covariance (11 bins here, bin0-bin10), matching the Kashur-style spline
#   method in your systematics writeup.
#
# This matters a lot for Anne's question. If the Cathode Bending variation
# only exists in this file as a histogram already summed over all events
# (vs. reco_leading_muon_ke), there is no way to re-bin it by proximity to
# cathode from this file alone — that requires per-event vertex/track
# position info for the *varied* sample, which may live in a different file
# from your detector-systematics production stage (before it gets
# histogrammed and written into `detsys_results`).
#
# This script does NOT assume an answer. It:
#   Part A — checks what these objects actually are (TTree vs TH1/TH2/other)
#   Part B — if `events/full/selected` is a real TTree, lists its branches
#            so we can see whether it has vertex/track position variables
#            at all (needed regardless of where the var09 sample lives)
#   Part C — if var09 turns out to be a real per-event tree, or if you find
#            the raw per-event variation file elsewhere, gives the binning
#            logic to actually test Anne's hypothesis.
#
# Run Part A + B first and send me the output before touching Part C.

# %%
import uproot
import numpy as np
import matplotlib.pyplot as plt

ROOT_FILE = (
    "/Users/rvizarreta/Library/CloudStorage/GoogleDrive-rvizarreta14@gmail.com/"
    "My Drive/🏛 PhD Repository/🚀 Research/🤖 Experiments&Projects/ICARUS/"
    "ICARUS_CC0pi_Selection/data/icarus_numi_numu_mc_onbeam_offbeam_syst_ppfx.root"
)

f = uproot.open(ROOT_FILE)

# %% [markdown]
# ## Part A — what kind of object is each relevant key?
# `classname` tells us TTree vs TH1D vs TH2D vs something else, for the
# handful of keys that matter for this question.

# %%
KEYS_TO_CHECK = [
    "events/full/selected",
    "events/onbeam/selected",
    "variations/reco_leading_muon_ke_var09_1D",
    "variations/reco_leading_muon_ke_var09_2D",
    "detsys_results/var09",
    "detsys_results/cv",
    "detsys_results/var09_splines/bin0",
]

for key in KEYS_TO_CHECK:
    try:
        obj = f[key]
        print(f"{key:55s} -> {type(obj).__name__}  (classname={obj.classname})")
    except Exception as e:
        print(f"{key:55s} -> ERROR: {e}")

# %% [markdown]
# ## Part B — branches of the real selected-events tree
# Using the corrected path `events/full/selected` (cycle 2, the latest). This
# only tells us about the CV/nominal sample, but we need to know whether
# vertex/track position variables exist here at all before worrying about
# where the var09 (Cathode Bending) sample's per-event info lives.

# %%
TREE_NAME = "events/full/selected"

tree = f[TREE_NAME]
print(f"Is this a TTree? {type(tree).__name__} / classname={tree.classname}")

if hasattr(tree, "keys"):
    all_branches = tree.keys()
    print(f"\nTotal branches: {len(all_branches)}")

    def grep_branches(patterns):
        hits = [b for b in all_branches if any(p.lower() in b.lower() for p in patterns)]
        return sorted(hits)

    print("\nPosition/geometry-related branches:")
    for b in grep_branches(["x", "vtx", "vertex", "drift", "tpc", "cryo", "cathode", "start", "end"]):
        print(" ", b)

    print("\nAll branches (for reference, in case the position variable is named "
          "something unexpected):")
    for b in sorted(all_branches):
        print(" ", b)
else:
    print("This key is not a TTree — see Part A classname output above.")

# %% [markdown]
# ## Part C-0 — if var09 is a histogram, look at its shape/axes directly
# This won't answer Anne's question, but it tells us exactly what was kept:
# one distribution (reco_leading_muon_ke) integrated over all events, with no
# position information retained. If that's the case, say so — don't try to
# force a per-event proximity plot out of a histogram that doesn't have one.

# %%
try:
    h = f["variations/reco_leading_muon_ke_var09_1D"]
    if "TH1" in h.classname:
        values, edges = h.to_numpy()
        print("var09_1D is a 1D histogram.")
        print(f"  {len(values)} bins, range [{edges[0]:.1f}, {edges[-1]:.1f}]")
    elif "TH2" in h.classname:
        values, xedges, yedges = h.to_numpy()
        print("var09_1D is (surprisingly) a 2D histogram.")
        print(f"  shape {values.shape}, x-range [{xedges[0]:.1f}, {xedges[-1]:.1f}], "
              f"y-range [{yedges[0]:.1f}, {yedges[-1]:.1f}]")
    else:
        print(f"Unexpected classname: {h.classname}")
except Exception as e:
    print(f"Could not inspect var09_1D: {e}")

try:
    h2 = f["variations/reco_leading_muon_ke_var09_2D"]
    print(f"\nvar09_2D classname: {h2.classname}")
    if "TH2" in h2.classname:
        values, xedges, yedges = h2.to_numpy()
        print(f"  shape {values.shape}, x-range [{xedges[0]:.1f}, {xedges[-1]:.1f}], "
              f"y-range [{yedges[0]:.1f}, {yedges[-1]:.1f}]")
except Exception as e:
    print(f"Could not inspect var09_2D: {e}")

# %% [markdown]
# ## What to do with this output
#
# Send me back:
# 1. The Part A classnames (TTree vs TH1D vs TH2D vs other) for each key.
# 2. The branch list from Part B, especially anything matching x/vtx/drift.
# 3. The bin counts/ranges from Part C-0.
#
# Depending on what comes back, there are two realistic paths forward:
#
# - **If `events/full/selected` has a usable vertex-x (or similar drift
#   coordinate) branch, but the Cathode Bending variation is only a
#   pre-summed histogram**: we cannot re-bin the existing var09 result by
#   proximity to cathode after the fact. The honest answer to Anne is that
#   this would require rerunning the Cathode Bending variation sample with
#   position information retained (or re-deriving it from whatever
#   pre-histogram file your detector-systematics pipeline produces, if one
#   exists) — this is worth saying plainly rather than forcing a plot from
#   data that doesn't support it.
# - **If there's a separate per-event file for the var09 production
#   (upstream of `detsys_results`)**: point me to it and I'll write the
#   actual proximity-to-cathode binning script against that file.

events/full/selected                                    -> Model_TTree_v20  (classname=TTree)
events/onbeam/selected                                  -> Model_TTree_v20  (classname=TTree)
variations/reco_leading_muon_ke_var09_1D                -> Model_TH1D_v3  (classname=TH1D)
variations/reco_leading_muon_ke_var09_2D                -> Model_TH2D_v4  (classname=TH2D)
detsys_results/var09                                    -> Model_TH1D_v3  (classname=TH1D)
detsys_results/cv                                       -> Model_TH1D_v3  (classname=TH1D)
detsys_results/var09_splines/bin0                       -> Model_TSpline3_v2  (classname=TSpline3)
Is this a TTree? Model_TTree_v20 / classname=TTree

Total branches: 128

Position/geometry-related branches:
  ppfx_cv_weight
  reco_leading_muon_end_x
  reco_leading_muon_end_y
  reco_leading_muon_end_z
  reco_vertex_x
  reco_vertex_y
  reco_vertex_z
  true_leading_muon_end_x
  true_leading_muon_end_y
  true_leading_muon_end_z
  true_vertex_x
  t